# Entrenamiento v2 — Mejorar precisión

**Modelo actual:** EfficientNet-B0, 84.89% Top-1

**Mejoras en esta versión:**
- Continuar entrenamiento desde el mejor modelo (resume)
- Label smoothing (0.1) — reduce overfitting
- Mixup (alpha=0.2) — regularización por mezcla de imágenes
- Random Erasing — fuerza al modelo a no depender de una sola zona
- 20 épocas adicionales de fine-tuning
- Batch size 64 con T4

**Instrucciones:**
1. `Entorno de ejecución → Cambiar tipo → T4 GPU`
2. Ejecuta todas las celdas
3. Descarga el modelo mejorado al final

In [ ]:
# 1. Verificar GPU
import torch
assert torch.cuda.is_available(), "GPU no detectada. Cambia el runtime a T4 GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 2. Clonar repo (con modelo actual incluido)
!git clone https://github.com/Fernando-Alvarado-Soria/app-caloriasv2.git
%cd app-caloriasv2

In [ ]:
# 3. Instalar dependencias
!pip install torch torchvision numpy --quiet

In [ ]:
# 4. Verificar que el modelo anterior existe para hacer resume
import os
model_path = 'ml/models/export/model_scripted.pt'
meta_path = 'ml/models/export/metadata.json'
print(f"model_scripted.pt: {'EXISTE' if os.path.exists(model_path) else 'NO ENCONTRADO'}")
print(f"metadata.json: {'EXISTE' if os.path.exists(meta_path) else 'NO ENCONTRADO'}")

if os.path.exists(meta_path):
    import json
    meta = json.load(open(meta_path))
    print(f"\nModelo actual: {meta['model_name']}")
    print(f"Accuracy actual: {meta['val_accuracy_top1']}% Top-1")
    print(f"Época: {meta['epoch']}")

In [ ]:
# 5. Configurar hiperparámetros mejorados
import ml.config as config

# Con T4 podemos usar batch más grande
config.BATCH_SIZE = 64
config.NUM_WORKERS = 2
config.DEVICE = "cuda"

# Más épocas de fine-tuning
config.NUM_EPOCHS = 20

# Ya estamos en fine-tuning, backbone descongelado desde el inicio
config.FREEZE_BACKBONE_EPOCHS = 0
config.UNFREEZE_AFTER = 0

# Learning rate más bajo para fine-tuning continuo
config.LEARNING_RATE = 5e-4
config.LR_BACKBONE = 5e-5

# Mejoras de regularización
config.LABEL_SMOOTHING = 0.1
config.MIXUP_ALPHA = 0.2
config.RANDOM_ERASING_PROB = 0.25
config.WEIGHT_DECAY = 1e-4

# Early stopping más paciente (entrenar más tiempo)
config.EARLY_STOP_PATIENCE = 6
config.CHECKPOINT_EVERY = 5

print("=== Configuración v2 ===")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Épocas adicionales: {config.NUM_EPOCHS}")
print(f"LR cabeza: {config.LEARNING_RATE}")
print(f"LR backbone: {config.LR_BACKBONE}")
print(f"Label smoothing: {config.LABEL_SMOOTHING}")
print(f"Mixup alpha: {config.MIXUP_ALPHA}")
print(f"Random erasing: {config.RANDOM_ERASING_PROB}")
print(f"Early stop patience: {config.EARLY_STOP_PATIENCE}")

In [ ]:
# 6. Reconstruir best_model.pt desde model_scripted.pt
# (El repo solo tiene el TorchScript exportado, necesitamos state_dict)
import torch
import torch.nn as nn
from ml.train import build_model, get_device

device = get_device()
print(f"Device: {device}")

# Cargar el modelo scripted y extraer los pesos
scripted = torch.jit.load('ml/models/export/model_scripted.pt', map_location=device)

# Crear modelo fresco y cargar pesos del scripted
model = build_model(device)
model.load_state_dict(scripted.state_dict())

# Guardar como checkpoint para resume
import os
os.makedirs('ml/models', exist_ok=True)
torch.save({
    'epoch': 9,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': {},
    'val_acc': 84.89,
}, 'ml/models/best_model.pt')
print("best_model.pt reconstruido para resume")

In [ ]:
# 7. Descargar dataset Food-101
from ml.download_dataset import download_food101
download_food101()

In [ ]:
# 8. ENTRENAR — continuar desde el modelo anterior
import argparse
from ml.train import train

train_args = argparse.Namespace(
    epochs=config.NUM_EPOCHS,
    batch_size=config.BATCH_SIZE,
    resume='auto',  # Carga best_model.pt y continua desde ahí
)

train(train_args)

In [ ]:
# 9. Evaluar modelo mejorado
!python -m ml.evaluate

In [ ]:
# 10. Exportar modelo mejorado
!python -m ml.export_model

In [ ]:
# 11. Comparar con modelo anterior
import json
meta = json.load(open('ml/models/export/metadata.json'))
print(f"\n{'='*50}")
print(f"RESULTADO FINAL")
print(f"{'='*50}")
print(f"Modelo anterior: 84.89% Top-1")
print(f"Modelo nuevo:    {meta['val_accuracy_top1']}% Top-1")
diff = meta['val_accuracy_top1'] - 84.89
print(f"Mejora:          {'+' if diff > 0 else ''}{diff:.2f}%")
print(f"Época final:     {meta['epoch']}")

In [ ]:
# 12. Descargar modelo mejorado
from google.colab import files
import shutil

os.makedirs('download_pack', exist_ok=True)
if os.path.exists('ml/models/export'):
    shutil.copytree('ml/models/export', 'download_pack/export', dirs_exist_ok=True)
if os.path.exists('ml/models/best_model.pt'):
    shutil.copy2('ml/models/best_model.pt', 'download_pack/best_model.pt')

shutil.make_archive('modelo_v2_mejorado', 'zip', 'download_pack')
files.download('modelo_v2_mejorado.zip')
print('Descarga iniciada. Copia los archivos a ml/models/export/ en tu proyecto.')